# 02. Khám Phá Dữ Liệu Có Mục Tiêu (Targeted Exploratory Data Analysis - EDA)
**Học phần:** Khai thác dữ liệu - Nhóm 12  
---
### Đối chiếu với Mục 1.2 - Pipeline bắt buộc của đồ án:
- [x] **Bước 5:** Khám phá dữ liệu (EDA) có mục tiêu: Dùng bảng thống kê và biểu đồ để hiểu dữ liệu, phân phối, tương quan, phát hiện ngoại lệ (outliers), chuẩn bị cơ sở cho việc scale và lựa chọn thuật toán.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

figures_dir = project_root / 'reports' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Thống kê mô tả và độ lệch phân phối (Skewness)
Phân tích trung bình, độ lệch chuẩn, khoảng tứ phân vị và hệ số lệch của 6 đặc trưng chính.

In [ ]:
df = pd.read_csv(project_root / 'data' / 'interim' / 'happiness_merged.csv')
feature_cols = ['gdp_per_capita', 'social_support', 'healthy_life_expectancy', 'freedom', 'generosity', 'corruption_perception']

stats_summary = df[feature_cols].describe().T
stats_summary['skewness'] = df[feature_cols].skew()
stats_summary

## 2. Trực quan hóa hình dạng phân phối (Histograms & KDE)
Kiểm tra xem đặc trưng nào có phân phối chuẩn, đặc trưng nào bị lệch mạnh (ví dụ corruption perception thường bị lệch phải mạnh).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(feature_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'Phân phối: {col} (Skew: {df[col].skew():.2f})')
plt.tight_layout()
plt.savefig(figures_dir / 'feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Phân tích tương quan giữa các đặc trưng và với Happiness Score
Đánh giá mức độ đa cộng tuyến (multicollinearity) và tương quan với điểm hạnh phúc thực tế.

In [ ]:
corr_cols = feature_cols + ['happiness_score']
corr_matrix = df[corr_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Ma Trận Tương Quan Giữa Các Yếu Tố Và Happiness Score', fontsize=14)
plt.savefig(figures_dir / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Phát hiện và đánh giá ngoại lệ (Outlier Detection qua Boxplots)
Xác định các giá trị ngoại lệ để quyết định sử dụng `StandardScaler` hay `RobustScaler`.

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df[feature_cols], palette='Set2')
plt.title('Biểu Đồ Hộp Nhận Diện Outliers Trên 6 Đặc Trưng', fontsize=14)
plt.xticks(rotation=15)
plt.savefig(figures_dir / 'boxplots_outliers.png', dpi=300, bbox_inches='tight')
plt.show()